In [1]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score,
    precision_score, recall_score, brier_score_loss,
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


In [2]:
def train_logistic_regression(data_dir="../data/processed", model_dir="../models"):
    print("--- Training Logistic Regression (SMOTE within CV folds) ---")

    # Load processed data (NO SMOTE at this stage; oversampling applied inside CV folds)
    X_train = pd.read_csv(os.path.join(data_dir, "X_train.csv"))
    y_train = pd.read_csv(os.path.join(data_dir, "y_train.csv")).values.ravel()
    X_test = pd.read_csv(os.path.join(data_dir, "X_test.csv"))
    y_test = pd.read_csv(os.path.join(data_dir, "y_test.csv")).values.ravel()

    # Hyperparameter grid (liblinear supports L1 + L2)
    param_grid = {
        'classifier__C': [0.001, 0.01, 0.1, 1, 10, 100],
        'classifier__penalty': ['l1', 'l2']
    }

    # Pipeline: SMOTE inside each CV fold only
    pipeline = ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('classifier', LogisticRegression(solver='liblinear', random_state=42, max_iter=1000))
    ])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
    grid_search.fit(X_train, y_train)

    best_lr = grid_search.best_estimator_
    print(f"Best Hyperparameters: {grid_search.best_params_}")

    # Evaluate on untouched hold-out test set
    y_test_probs = best_lr.predict_proba(X_test)[:, 1]
    y_test_preds = best_lr.predict(X_test)

    print("\nTest Metrics:")
    print(f"  AUC-ROC:        {roc_auc_score(y_test, y_test_probs):.4f}")
    print(f"  F1-Score:       {f1_score(y_test, y_test_preds):.4f}")
    print(f"  Accuracy:       {accuracy_score(y_test, y_test_preds):.4f}")
    print(f"  Precision:      {precision_score(y_test, y_test_preds):.4f}")
    print(f"  Recall (Sens):  {recall_score(y_test, y_test_preds):.4f}")
    print(f"  Brier Score:    {brier_score_loss(y_test, y_test_probs):.4f}")

    # Extract coefficients from the classifier step for global interpretability
    coefficients = best_lr.named_steps['classifier'].coef_[0]
    odds_ratios = np.exp(coefficients)

    features_df = pd.DataFrame({
        'Feature': X_train.columns,
        'Coefficient': coefficients,
        'Odds_Ratio': odds_ratios
    }).sort_values(by='Odds_Ratio', ascending=False)

    print("\nTop 5 Features Increasing Readmission Risk (Odds Ratio > 1):")
    print(features_df.head(5).to_string(index=False))
    print("\nTop 5 Features Decreasing Readmission Risk (Odds Ratio < 1):")
    print(features_df.tail(5).to_string(index=False))

    # Serialize
    os.makedirs(model_dir, exist_ok=True)
    model_path = os.path.join(model_dir, "logistic_regression.pkl")
    joblib.dump(best_lr, model_path)
    print(f"\nModel artifact serialized to: {model_path}\n")

In [3]:
train_logistic_regression()

--- Training Logistic Regression (SMOTE within CV folds) ---


Best Hyperparameters: {'classifier__C': 0.01, 'classifier__penalty': 'l1'}

Test Metrics:
  AUC-ROC:        0.6424
  F1-Score:       0.2642
  Accuracy:       0.6378
  Precision:      0.1682
  Recall (Sens):  0.6159
  Brier Score:    0.2314

Top 5 Features Increasing Readmission Risk (Odds Ratio > 1):
                  Feature  Coefficient  Odds_Ratio
         number_inpatient     0.365901    1.441812
         number_diagnoses     0.260791    1.297957
          num_medications     0.216899    1.242218
       payer_code_Unknown     0.114688    1.121523
medical_specialty_Unknown     0.043890    1.044868

Top 5 Features Decreasing Readmission Risk (Odds Ratio < 1):
          Feature  Coefficient  Odds_Ratio
        change_Ch    -0.027443    0.972930
number_outpatient    -0.027815    0.972568
      gender_Male    -0.084067    0.919369
  diabetesMed_Yes    -0.086742    0.916914
     A1Cresult_>8    -0.338460    0.712868

Model artifact serialized to: ../models/logistic_regression.pkl



In [4]:
def train_decision_tree(data_dir="../data/processed", model_dir="../models"):
    print("--- Training Decision Tree (SMOTE within CV folds) ---")

    X_train = pd.read_csv(os.path.join(data_dir, "X_train.csv"))
    y_train = pd.read_csv(os.path.join(data_dir, "y_train.csv")).values.ravel()
    X_test = pd.read_csv(os.path.join(data_dir, "X_test.csv"))
    y_test = pd.read_csv(os.path.join(data_dir, "y_test.csv")).values.ravel()

    param_grid = {
        'classifier__max_depth': [4, 5, 6, 7, 8],
        'classifier__criterion': ['gini', 'entropy'],
        'classifier__min_samples_split': [10, 20, 50]
    }

    pipeline = ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('classifier', DecisionTreeClassifier(random_state=42))
    ])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
    grid_search.fit(X_train, y_train)

    best_dt = grid_search.best_estimator_
    print(f"Best Hyperparameters: {grid_search.best_params_}")

    y_test_probs = best_dt.predict_proba(X_test)[:, 1]
    y_test_preds = best_dt.predict(X_test)

    print("\nTest Metrics:")
    print(f"  AUC-ROC:        {roc_auc_score(y_test, y_test_probs):.4f}")
    print(f"  F1-Score:       {f1_score(y_test, y_test_preds):.4f}")
    print(f"  Accuracy:       {accuracy_score(y_test, y_test_preds):.4f}")
    print(f"  Precision:      {precision_score(y_test, y_test_preds):.4f}")
    print(f"  Recall (Sens):  {recall_score(y_test, y_test_preds):.4f}")
    print(f"  Brier Score:    {brier_score_loss(y_test, y_test_probs):.4f}")

    # Feature importance from the classifier step
    importances = best_dt.named_steps['classifier'].feature_importances_
    features_df = pd.DataFrame({
        'Feature': X_train.columns,
        'Gini_Importance': importances
    }).sort_values(by='Gini_Importance', ascending=False)

    print("\nTop 5 Most Informative Splits (Gini Importance):")
    print(features_df.head(5).to_string(index=False))

    os.makedirs(model_dir, exist_ok=True)
    model_path = os.path.join(model_dir, "decision_tree.pkl")
    joblib.dump(best_dt, model_path)
    print(f"\nModel artifact serialized to: {model_path}\n")

In [5]:
train_decision_tree()

--- Training Decision Tree (SMOTE within CV folds) ---


Best Hyperparameters: {'classifier__criterion': 'entropy', 'classifier__max_depth': 6, 'classifier__min_samples_split': 20}

Test Metrics:
  AUC-ROC:        0.6167
  F1-Score:       0.1899
  Accuracy:       0.8210
  Precision:      0.1818
  Recall (Sens):  0.1987
  Brier Score:    0.1336

Top 5 Most Informative Splits (Gini Importance):
                  Feature  Gini_Importance
         number_inpatient         0.458312
 discharge_disposition_id         0.255337
       diag_3_group_Other         0.067840
             A1Cresult_>8         0.063034
medical_specialty_Unknown         0.044284

Model artifact serialized to: ../models/decision_tree.pkl



In [6]:
def train_random_forest(data_dir="../data/processed", model_dir="../models"):
    print("--- Training Random Forest (SMOTE within CV folds) ---")

    X_train = pd.read_csv(os.path.join(data_dir, "X_train.csv"))
    y_train = pd.read_csv(os.path.join(data_dir, "y_train.csv")).values.ravel()
    X_test = pd.read_csv(os.path.join(data_dir, "X_test.csv"))
    y_test = pd.read_csv(os.path.join(data_dir, "y_test.csv")).values.ravel()

    param_grid = {
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [10, 15, 20],
        'classifier__min_samples_split': [5, 10],
        'classifier__max_features': ['sqrt', 'log2']
    }

    pipeline = ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1))
    ])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
    grid_search.fit(X_train, y_train)

    best_rf = grid_search.best_estimator_
    print(f"Best Hyperparameters: {grid_search.best_params_}")

    y_test_probs = best_rf.predict_proba(X_test)[:, 1]
    y_test_preds = best_rf.predict(X_test)

    print("\nTest Metrics:")
    print(f"  AUC-ROC:        {roc_auc_score(y_test, y_test_probs):.4f}")
    print(f"  F1-Score:       {f1_score(y_test, y_test_preds):.4f}")
    print(f"  Accuracy:       {accuracy_score(y_test, y_test_preds):.4f}")
    print(f"  Precision:      {precision_score(y_test, y_test_preds):.4f}")
    print(f"  Recall (Sens):  {recall_score(y_test, y_test_preds):.4f}")
    print(f"  Brier Score:    {brier_score_loss(y_test, y_test_probs):.4f}")

    importances = best_rf.named_steps['classifier'].feature_importances_
    features_df = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)

    print("\nTop 5 Gini Importance Features:")
    print(features_df.head(5).to_string(index=False))

    os.makedirs(model_dir, exist_ok=True)
    model_path = os.path.join(model_dir, "random_forest.pkl")
    joblib.dump(best_rf, model_path)
    print(f"\nModel artifact serialized to: {model_path}\n")

In [7]:
train_random_forest()

--- Training Random Forest (SMOTE within CV folds) ---


Best Hyperparameters: {'classifier__max_depth': 10, 'classifier__max_features': 'sqrt', 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200}

Test Metrics:
  AUC-ROC:        0.6590
  F1-Score:       0.1485
  Accuracy:       0.8797
  Precision:      0.2941
  Recall (Sens):  0.0993
  Brier Score:    0.1217

Top 5 Gini Importance Features:
                  Feature  Importance
         number_inpatient    0.090627
 discharge_disposition_id    0.056694
         number_diagnoses    0.047814
           A1Cresult_None    0.035310
medical_specialty_Unknown    0.034127

Model artifact serialized to: ../models/random_forest.pkl



In [8]:
def train_xgboost(data_dir="../data/processed", model_dir="../models"):
    print("--- Training XGBoost (SMOTE within CV folds) ---")

    X_train = pd.read_csv(os.path.join(data_dir, "X_train.csv"))
    y_train = pd.read_csv(os.path.join(data_dir, "y_train.csv")).values.ravel()
    X_test = pd.read_csv(os.path.join(data_dir, "X_test.csv"))
    y_test = pd.read_csv(os.path.join(data_dir, "y_test.csv")).values.ravel()

    # Sanitize feature names for XGBoost C++ backend
    for df in [X_train, X_test]:
        df.columns = (df.columns
                      .str.replace('[', '_', regex=False)
                      .str.replace(']', '_', regex=False)
                      .str.replace('<', 'lt_', regex=False)
                      .str.replace('>', 'gt_', regex=False))

    param_grid = {
        'classifier__max_depth': [4, 6, 8],
        'classifier__learning_rate': [0.01, 0.1, 0.2],
        'classifier__n_estimators': [100, 200],
        'classifier__reg_alpha': [0.1, 1.0],
        'classifier__reg_lambda': [1.0, 5.0]
    }

    pipeline = ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('classifier', XGBClassifier(
            tree_method='hist',
            random_state=42,
            eval_metric='logloss',
            n_jobs=-1
        ))
    ])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
    grid_search.fit(X_train, y_train)

    best_xgb = grid_search.best_estimator_
    print(f"Best Hyperparameters: {grid_search.best_params_}")

    y_test_probs = best_xgb.predict_proba(X_test)[:, 1]
    y_test_preds = best_xgb.predict(X_test)

    print("\nTest Metrics:")
    print(f"  AUC-ROC:        {roc_auc_score(y_test, y_test_probs):.4f}")
    print(f"  F1-Score:       {f1_score(y_test, y_test_preds):.4f}")
    print(f"  Accuracy:       {accuracy_score(y_test, y_test_preds):.4f}")
    print(f"  Precision:      {precision_score(y_test, y_test_preds):.4f}")
    print(f"  Recall (Sens):  {recall_score(y_test, y_test_preds):.4f}")
    print(f"  Brier Score:    {brier_score_loss(y_test, y_test_probs):.4f}")

    importances = best_xgb.named_steps['classifier'].feature_importances_
    features_df = pd.DataFrame({
        'Feature': X_train.columns,
        'Gain_Importance': importances
    }).sort_values(by='Gain_Importance', ascending=False)

    print("\nTop 5 Gain Importance Features:")
    print(features_df.head(5).to_string(index=False))

    os.makedirs(model_dir, exist_ok=True)
    model_path = os.path.join(model_dir, "xgboost.pkl")
    joblib.dump(best_xgb, model_path)
    print(f"\nModel artifact serialized to: {model_path}\n")

In [9]:
train_xgboost()

--- Training XGBoost (SMOTE within CV folds) ---


Best Hyperparameters: {'classifier__learning_rate': 0.01, 'classifier__max_depth': 4, 'classifier__n_estimators': 200, 'classifier__reg_alpha': 0.1, 'classifier__reg_lambda': 1.0}

Test Metrics:
  AUC-ROC:        0.6438
  F1-Score:       0.1579
  Accuracy:       0.8434
  Precision:      0.1826
  Recall (Sens):  0.1391
  Brier Score:    0.1322

Top 5 Gain Importance Features:
                 Feature  Gain_Importance
             gender_Male         0.074708
          A1Cresult_None         0.068727
        number_inpatient         0.065379
      diag_3_group_Other         0.058361
discharge_disposition_id         0.055959

Model artifact serialized to: ../models/xgboost.pkl

